In [ ]:
from __future__ import annotations

import json
import logging
from pathlib import Path
from typing import Any
from src.config_loader import CFG, ROOT_DIR

In [ ]:
logger = logging.getLogger(__name__)

Dataset = list[dict[str, Any]]

In [ ]:
def _read_jsonl(path: Path) -> Dataset:
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found:\n{path}"
        )

    rows: Dataset = []

    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(
                    f"Invalid JSON in {path} "
                    f"(line {line_no})"
                ) from e

    if not rows:
        raise ValueError(
            f"{path} contains no dataset entries."
        )

    logger.info("Loaded %d records from %s", len(rows), path.name)

    return rows

In [ ]:
def _dataset_path(dataset_name: str, split: str) -> Path:
    cfg = getattr(CFG.datasets, dataset_name)
    return ROOT_DIR / Path(cfg.path) / f"{split}.jsonl"

In [ ]:
def load_codocbench(split: str | None = None) -> Dataset:
    split = split or CFG.datasets.codocbench.split
    return _read_jsonl(_dataset_path("codocbench", split))

In [ ]:
def load_spider(split: str | None = None) -> Dataset:
    split = split or CFG.datasets.spider.split
    return _read_jsonl(_dataset_path("spider", split))

In [ ]:
def load_birdbench(split: str | None = None) -> Dataset:
    cfg_name = (
        "birdbench"
        if hasattr(CFG.datasets, "birdbench")
        else "birdBench"
    )

    split = split or getattr(CFG.datasets, cfg_name).split
    return _read_jsonl(_dataset_path(cfg_name, split))

In [ ]:
def load_raw_language_corpus(language: str) -> Dataset:
    path = ROOT_DIR / "data" / f"{language.lower()}_raw.jsonl"
    return _read_jsonl(path)